# 02 Pose-Guided Warp And Paste-Back

This notebook takes the `rep` foreground exported by `01`, aligns it to the dog pose and location in `org` using **keypoint-driven geometry**, and pastes the warped dog back into the original scene to create a seed composite for notebook `03`.

The key goals of this step are:

1. determine whether `org` and `rep` are geometrically compatible enough to align
2. allow a mirrored `rep` if both images are side profiles with opposite left/right direction
3. build explicit source/target correspondences from shared keypoints
4. use a similarity transform as a stable global initialization
5. refine the fit with a piecewise-affine warp
6. export the inputs needed by `03`:
   - `seed_composite.png`
   - `warped_mask.png`
   - `refinement_mask.png`

Diffusion is intentionally postponed to the next notebook. The purpose of `02` is not image generation. It is to make the **geometric alignment step visible and explainable**.


## Step 04 - Reinsert True-DragGAN Face, Then Body/Leg Warp

This notebook loads the face edited by real DragGAN in notebook `03`, pastes it back into the replacement cutout, and then runs the old conservative body/leg pose matching.

Important behavior change: face/head keypoints are excluded from the body warp. The face has already been handled by GAN latent editing, so the geometric warp is only allowed to align torso, tail, and legs.

In [ ]:
%matplotlib inline


## Input Contract

This notebook now processes up to **5 example pairs** in sequence.

It reads the cutout packages produced by notebook `01` for the discovered example pairs and writes one warp package per pair to:

- `data/outputs/02_pose_warp/<pair_name>/`


In [ ]:
import json
import math
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from skimage.transform import PiecewiseAffineTransform, SimilarityTransform, warp
from ultralytics import YOLO


def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data" / "outputs" / "face_draggan_pipeline" / "01_cutout").exists() and (candidate / "notebooks").exists():
            return candidate
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "environment.yml").exists() or (candidate / ".git").exists():
            return candidate
    return cwd


def resolve_input_image(label, base_name, env_key, example_dir):
    if os.environ.get(env_key):
        path = Path(os.environ[env_key]).expanduser().resolve()
        if path.exists():
            return path
        raise FileNotFoundError(f"{env_key} is set but does not exist: {path}")
    for suffix in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
        candidate = example_dir / f"{base_name}{suffix}"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"{label} not found. Put it under {example_dir} using base name {base_name}, or set {env_key}."
    )


def resolve_pose_model_path(project_root):
    candidates = [
        Path(os.environ["DOG_POSE_MODEL_PATH"]) if os.environ.get("DOG_POSE_MODEL_PATH") else None,
        project_root / "models" / "dog_pose_best.pt",
        project_root / "dog_pose_best.pt",
        project_root / "data" / "outputs" / "dog_pose_training" / "dog_pose_bootstrap" / "weights" / "best.pt",
    ]
    for candidate in candidates:
        if candidate is not None and candidate.exists():
            return candidate
    return project_root / "models" / "dog_pose_best.pt"


def resolve_example_image(example_dir, stem):
    for suffix in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
        candidate = example_dir / f"{stem}{suffix}"
        if candidate.exists():
            return candidate
    return None


def discover_example_pairs(example_dir, max_pairs=5):
    if os.environ.get("DOG_REPLACEMENT_ORG_PATH") and os.environ.get("DOG_REPLACEMENT_REP_PATH"):
        org_path = Path(os.environ["DOG_REPLACEMENT_ORG_PATH"]).expanduser().resolve()
        rep_path = Path(os.environ["DOG_REPLACEMENT_REP_PATH"]).expanduser().resolve()
        if not org_path.exists() or not rep_path.exists():
            raise FileNotFoundError("Custom environment-path pair does not exist.")
        return [{"pair_id": "custom", "org_path": org_path, "rep_path": rep_path}]

    pair_specs = []
    for idx in range(1, max_pairs + 1):
        org_path = resolve_example_image(example_dir, f"example_{idx:02d}_org")
        rep_path = resolve_example_image(example_dir, f"example_{idx:02d}_rep")
        if org_path is None and rep_path is None:
            continue
        if org_path is None or rep_path is None:
            continue
        pair_specs.append({"pair_id": f"{idx:02d}", "org_path": org_path, "rep_path": rep_path})

    if pair_specs:
        return pair_specs

    org_path = resolve_input_image("Original image", "example_org", "DOG_REPLACEMENT_ORG_PATH", example_dir)
    rep_path = resolve_input_image("Replacement image", "example_rep", "DOG_REPLACEMENT_REP_PATH", example_dir)
    return [{"pair_id": "legacy", "org_path": org_path, "rep_path": rep_path}]


def build_pair_name(org_path, rep_path):
    return f"{org_path.stem}__{rep_path.stem}"


def print_pair_header(title):
    print("\n" + "=" * 96)
    print(title)
    print("=" * 96)


if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires a local CUDA GPU, but no CUDA device is available.")

PROJECT_ROOT = find_project_root()
EXAMPLE_DIR = PROJECT_ROOT / "data" / "example"
CUTOUT_ROOT = PROJECT_ROOT / "data" / "outputs" / "face_draggan_pipeline" / "01_cutout"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "outputs" / "face_draggan_pipeline" / "04_body_warp_after_true_draggan"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FACE_EDIT_ROOT = PROJECT_ROOT / "data" / "outputs" / "face_draggan_pipeline" / "03_true_draggan_face_edit"
ACTIVE_CUTOUT_MANIFEST_PATH = CUTOUT_ROOT / "_active_batch_manifest.json"
ACTIVE_WARP_MANIFEST_PATH = OUTPUT_ROOT / "_active_batch_manifest.json"
POSE_MODEL_PATH = resolve_pose_model_path(PROJECT_ROOT)
DEVICE = 0
ENABLE_MICRO_PIECEWISE_WARP = os.environ.get("DOG_ENABLE_MICRO_PIECEWISE_WARP", "True").lower() in {"1", "true", "yes", "on"}
ENABLE_FULL_PIECEWISE_WARP = os.environ.get("DOG_ENABLE_FULL_PIECEWISE_WARP", "False").lower() in {"1", "true", "yes", "on"}
BODY_ALIGNMENT_MODE = os.environ.get("DOG_TRUE_DRAGGAN_BODY_ALIGNMENT_MODE", "body_warp")
VALID_BODY_ALIGNMENT_MODES = {"body_warp", "face_only_naive_box"}
if BODY_ALIGNMENT_MODE not in VALID_BODY_ALIGNMENT_MODES:
    raise ValueError(f"Unknown DOG_TRUE_DRAGGAN_BODY_ALIGNMENT_MODE={BODY_ALIGNMENT_MODE}. Valid options: {sorted(VALID_BODY_ALIGNMENT_MODES)}")
MICRO_PIECEWISE_MAX_EXTRA_REL = float(os.environ.get("DOG_MICRO_PIECEWISE_MAX_EXTRA_REL", "0.035"))
MICRO_PIECEWISE_MIN_EXTRA_PX = float(os.environ.get("DOG_MICRO_PIECEWISE_MIN_EXTRA_PX", "6.0"))
ENABLE_REPLACEMENT_SIZE_PREMATCH = os.environ.get("DOG_ENABLE_REPLACEMENT_SIZE_PREMATCH", "True").lower() in {"1", "true", "yes", "on"}
REPLACEMENT_SIZE_PREMATCH_MIN_SCALE = float(os.environ.get("DOG_REPLACEMENT_SIZE_PREMATCH_MIN_SCALE", "0.40"))
REPLACEMENT_SIZE_PREMATCH_MAX_SCALE = float(os.environ.get("DOG_REPLACEMENT_SIZE_PREMATCH_MAX_SCALE", "2.50"))
REPLACEMENT_DESTINATION_SCALE = float(os.environ.get("DOG_REPLACEMENT_DESTINATION_SCALE", "1.03"))
BACKGROUND_HOLE_EXPANSION_PX = int(os.environ.get("DOG_BACKGROUND_HOLE_EXPANSION_PX", "8"))

if not POSE_MODEL_PATH.exists():
    raise FileNotFoundError(f"Dog pose checkpoint not found at {POSE_MODEL_PATH}.")


def load_json_if_exists(path):
    path = Path(path)
    if not path.exists():
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


PAIR_JOBS = []
active_manifest = load_json_if_exists(ACTIVE_CUTOUT_MANIFEST_PATH)
if active_manifest:
    for idx, item in enumerate(active_manifest, start=1):
        metadata_path = PROJECT_ROOT / item["metadata_path"]
        pair_name = item["pair_name"]
        PAIR_JOBS.append(
            {
                "pair_id": item.get("pair_id", f"active_{idx:02d}"),
                "pair_name": pair_name,
                "org_path": None,
                "rep_path": None,
                "metadata_path": metadata_path,
                "output_dir": OUTPUT_ROOT / pair_name,
            }
        )
    ACTIVE_INPUT_MODE = "follow_notebook_01_active_output"
else:
    pair_specs = discover_example_pairs(EXAMPLE_DIR, max_pairs=5)
    for spec in pair_specs:
        pair_name = build_pair_name(spec["org_path"], spec["rep_path"])
        metadata_path = CUTOUT_ROOT / pair_name / "metadata.json"
        if not metadata_path.exists():
            raise FileNotFoundError(f"Cutout metadata not found at {metadata_path}. Run notebook 01 first.")
        PAIR_JOBS.append(
            {
                "pair_id": spec["pair_id"],
                "pair_name": pair_name,
                "org_path": spec["org_path"],
                "rep_path": spec["rep_path"],
                "metadata_path": metadata_path,
                "output_dir": OUTPUT_ROOT / pair_name,
            }
        )
    ACTIVE_INPUT_MODE = "fallback_example_pairs"

pose_model = YOLO(str(POSE_MODEL_PATH))
model_kpt_shape = getattr(getattr(pose_model, "model", None), "kpt_shape", None)
if model_kpt_shape is not None and int(model_kpt_shape[0]) != 24:
    raise RuntimeError(f"Expected a 24-keypoint dog pose checkpoint, but got {model_kpt_shape}.")

print("Project root:", PROJECT_ROOT)
print("Output root:", OUTPUT_ROOT)
print("True DragGAN face edit root:", FACE_EDIT_ROOT)
print("Pose model:", POSE_MODEL_PATH)
print("CUDA device:", torch.cuda.get_device_name(0))
print("Replacement size pre-match enabled:", ENABLE_REPLACEMENT_SIZE_PREMATCH)
print("Replacement size pre-match scale limits:", REPLACEMENT_SIZE_PREMATCH_MIN_SCALE, REPLACEMENT_SIZE_PREMATCH_MAX_SCALE)
print("Replacement destination scale:", REPLACEMENT_DESTINATION_SCALE)
print("Background hole expansion px:", BACKGROUND_HOLE_EXPANSION_PX)
print("Body alignment mode:", BODY_ALIGNMENT_MODE)
print("Enable micro-piecewise warp:", ENABLE_MICRO_PIECEWISE_WARP)
print("Enable full piecewise warp:", ENABLE_FULL_PIECEWISE_WARP)
print("Micro-piecewise max extra rel:", MICRO_PIECEWISE_MAX_EXTRA_REL)
print("Input mode:", ACTIVE_INPUT_MODE)
print("Discovered pair jobs:")
for job in PAIR_JOBS:
    print(" -", job["pair_id"], "|", job["pair_name"])


## Load The Cutout Assets From 01

At this point we load the assets needed for geometric alignment:

- the original image
- the original background with a hole
- the replacement image
- the replacement RGBA foreground and mask

From here onward, the question is no longer "how do we segment the dog?" The question becomes "how do we align the dog to the target skeleton and position?"


In [ ]:
def read_rgb(path):
    return np.array(Image.open(path).convert("RGB"))


def read_rgba(path):
    return np.array(Image.open(path).convert("RGBA"))


def read_mask(path):
    return np.array(Image.open(path).convert("L")) > 127


def save_rgb(path, image):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(np.clip(image, 0, 255).astype(np.uint8)).save(path)


def save_mask(path, mask):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((mask.astype(np.uint8) * 255)).save(path)


def ensure_uint8(image):
    return np.clip(image, 0, 255).astype(np.uint8)


def plot_images(items, cols=3, figsize=(16, 10)):
    if not items:
        return
    rows = int(math.ceil(len(items) / cols))
    plt.figure(figsize=figsize)
    for idx, (title, image) in enumerate(items, start=1):
        plt.subplot(rows, cols, idx)
        if image.ndim == 2:
            plt.imshow(image, cmap="gray")
        else:
            plt.imshow(image)
        plt.title(title)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


def overlay_mask(image, mask, color=(30, 144, 255), alpha=0.45):
    out = image.astype(np.float32).copy()
    color_arr = np.array(color, dtype=np.float32)
    out[mask] = out[mask] * (1.0 - alpha) + color_arr * alpha
    return ensure_uint8(out)


def rgba_on_checkerboard(rgba):
    h, w = rgba.shape[:2]
    tile = 16
    yy, xx = np.indices((h, w))
    board = ((yy // tile + xx // tile) % 2).astype(np.uint8)
    checker = np.where(board[..., None] == 0, 220, 180).astype(np.uint8)
    checker = np.repeat(checker[..., :1], 3, axis=2)
    alpha = rgba[..., 3:4].astype(np.float32) / 255.0
    return ensure_uint8(rgba[..., :3].astype(np.float32) * alpha + checker.astype(np.float32) * (1.0 - alpha))


## Pose Helpers: Keypoint Detection, Mirroring Logic, And Skeleton Visualization

The core logic in this section is:

- if `org` and `rep` are both side-profile dogs but face opposite directions, mirroring `rep` is allowed
- if the view is frontal/rear or too few keypoints are reliable, the notebook should say clearly that the geometric basis is weak

In other words, this stage is about **explaining alignability**, not forcing a warp no matter what.


In [ ]:
DOG_KEYPOINT_NAMES = [
    "front_left_paw",
    "front_left_knee",
    "front_left_elbow",
    "rear_left_paw",
    "rear_left_knee",
    "rear_left_elbow",
    "front_right_paw",
    "front_right_knee",
    "front_right_elbow",
    "rear_right_paw",
    "rear_right_knee",
    "rear_right_elbow",
    "tail_start",
    "tail_end",
    "left_ear_base",
    "right_ear_base",
    "nose",
    "chin",
    "left_ear_tip",
    "right_ear_tip",
    "left_eye",
    "right_eye",
    "withers",
    "throat",
]
NAME_TO_INDEX = {name: idx for idx, name in enumerate(DOG_KEYPOINT_NAMES)}
KEYPOINT_CONF = 0.20
POSE_CONF = 0.15
POSE_IMGSZ = 960

DOG_SKELETON_EDGES = [
    ("nose", "chin"),
    ("nose", "left_eye"),
    ("nose", "right_eye"),
    ("left_eye", "left_ear_base"),
    ("right_eye", "right_ear_base"),
    ("left_ear_base", "left_ear_tip"),
    ("right_ear_base", "right_ear_tip"),
    ("throat", "nose"),
    ("throat", "withers"),
    ("withers", "tail_start"),
    ("tail_start", "tail_end"),
    ("withers", "front_left_elbow"),
    ("front_left_elbow", "front_left_knee"),
    ("front_left_knee", "front_left_paw"),
    ("withers", "front_right_elbow"),
    ("front_right_elbow", "front_right_knee"),
    ("front_right_knee", "front_right_paw"),
    ("tail_start", "rear_left_elbow"),
    ("rear_left_elbow", "rear_left_knee"),
    ("rear_left_knee", "rear_left_paw"),
    ("tail_start", "rear_right_elbow"),
    ("rear_right_elbow", "rear_right_knee"),
    ("rear_right_knee", "rear_right_paw"),
]


def clip_bbox(bbox, width, height):
    x1, y1, x2, y2 = bbox
    x1 = int(max(0, min(width - 1, math.floor(x1))))
    y1 = int(max(0, min(height - 1, math.floor(y1))))
    x2 = int(max(x1 + 1, min(width, math.ceil(x2))))
    y2 = int(max(y1 + 1, min(height, math.ceil(y2))))
    return [x1, y1, x2, y2]


def get_keypoint_arrays(result, instance_idx):
    xy = result.keypoints.xy[instance_idx].detach().cpu().numpy().astype(np.float32)
    conf_tensor = getattr(result.keypoints, "conf", None)
    if conf_tensor is not None:
        conf = conf_tensor[instance_idx].detach().cpu().numpy().astype(np.float32)
    else:
        data = result.keypoints.data[instance_idx].detach().cpu().numpy().astype(np.float32)
        conf = data[:, 2] if data.shape[-1] >= 3 else np.ones(len(xy), dtype=np.float32)
    return xy, conf


def named_point(points, confs, name, threshold=KEYPOINT_CONF):
    idx = NAME_TO_INDEX[name]
    if confs[idx] >= threshold:
        return points[idx]
    return None


def count_visible(points, confs, names, threshold=KEYPOINT_CONF):
    return sum(1 for name in names if named_point(points, confs, name, threshold) is not None)


def orientation_label(points, confs):
    nose = named_point(points, confs, "nose")
    tail_start = named_point(points, confs, "tail_start")
    if nose is not None and tail_start is not None:
        return "right" if nose[0] > tail_start[0] else "left"
    left_eye = named_point(points, confs, "left_eye")
    right_eye = named_point(points, confs, "right_eye")
    if left_eye is not None and right_eye is not None:
        return "frontal_or_rear"
    return "unknown"


def classify_view(points, confs):
    orientation = orientation_label(points, confs)
    left_eye = named_point(points, confs, "left_eye")
    right_eye = named_point(points, confs, "right_eye")
    if orientation in {"left", "right"}:
        return "side_profile"
    if left_eye is not None and right_eye is not None:
        return "frontal_or_rear"
    return "ambiguous"


def choose_best_instance(result):
    if result.boxes is None or result.keypoints is None or len(result.boxes) == 0:
        return None
    candidates = []
    for idx, box in enumerate(result.boxes):
        xyxy = box.xyxy[0].detach().cpu().numpy().astype(np.float32)
        area = max(1.0, (xyxy[2] - xyxy[0]) * (xyxy[3] - xyxy[1]))
        score = float(box.conf.item())
        visible = float((get_keypoint_arrays(result, idx)[1] >= KEYPOINT_CONF).sum())
        candidates.append((idx, score * math.sqrt(area) * (1.0 + visible / 24.0)))
    return max(candidates, key=lambda item: item[1])[0]


def detect_pose_record(image):
    results = pose_model.predict(image, conf=POSE_CONF, imgsz=POSE_IMGSZ, device=DEVICE, verbose=False)
    result = results[0]
    best_idx = choose_best_instance(result)
    if best_idx is None:
        raise RuntimeError("Pose model did not return a usable dog instance.")
    bbox = clip_bbox(result.boxes[best_idx].xyxy[0].detach().cpu().numpy().tolist(), image.shape[1], image.shape[0])
    keypoints_xy, keypoints_conf = get_keypoint_arrays(result, best_idx)
    return {
        "bbox_xyxy": bbox,
        "detection_conf": round(float(result.boxes[best_idx].conf.item()), 6),
        "points": keypoints_xy,
        "confs": keypoints_conf,
        "visible_keypoints": int((keypoints_conf >= KEYPOINT_CONF).sum()),
        "orientation": orientation_label(keypoints_xy, keypoints_conf),
        "view_class": classify_view(keypoints_xy, keypoints_conf),
    }


def mirror_points_x(points, width):
    mirrored = points.copy()
    mirrored[:, 0] = (width - 1) - mirrored[:, 0]
    return mirrored


def draw_pose_overlay(image, pose_record, show_bbox=True, title="Pose overlay"):
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(image)
    if show_bbox:
        x1, y1, x2, y2 = pose_record["bbox_xyxy"]
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2))
    point_map = {
        name: pt
        for name, pt, conf in zip(DOG_KEYPOINT_NAMES, pose_record["points"], pose_record["confs"])
        if conf >= KEYPOINT_CONF
    }
    for start_name, end_name in DOG_SKELETON_EDGES:
        start = point_map.get(start_name)
        end = point_map.get(end_name)
        if start is None or end is None:
            continue
        ax.plot([start[0], end[0]], [start[1], end[1]], color="cyan", linewidth=2)
    for pt in point_map.values():
        ax.scatter(pt[0], pt[1], s=24, color="yellow", edgecolors="black", linewidths=0.5)
    ax.set_title(
        f"{title}\n"
        f"{pose_record['view_class']} | {pose_record['orientation']} | visible_kp={pose_record['visible_keypoints']}"
    )
    ax.axis("off")
    plt.tight_layout()
    plt.show()



def deserialize_pose_record(data):
    if not data:
        return None
    return {
        "bbox_xyxy": [int(v) for v in data["bbox_xyxy"]],
        "detection_conf": float(data["detection_conf"]),
        "visible_keypoints": int(data["visible_keypoints"]),
        "orientation": data["orientation"],
        "view_class": data["view_class"],
        "points": np.array(data["points"], dtype=np.float32),
        "confs": np.array(data["confs"], dtype=np.float32),
    }


def translate_pose_record(pose_record, dx=0.0, dy=0.0):
    translated = {
        **pose_record,
        "bbox_xyxy": [
            int(round(pose_record["bbox_xyxy"][0] + dx)),
            int(round(pose_record["bbox_xyxy"][1] + dy)),
            int(round(pose_record["bbox_xyxy"][2] + dx)),
            int(round(pose_record["bbox_xyxy"][3] + dy)),
        ],
        "points": pose_record["points"].copy(),
        "confs": pose_record["confs"].copy(),
    }
    translated["points"][:, 0] += float(dx)
    translated["points"][:, 1] += float(dy)
    return translated


def pose_rmse(src_pts, dst_pts, tform):
    predicted = tform(src_pts)
    return float(np.sqrt(np.mean(np.sum((predicted - dst_pts) ** 2, axis=1))))


def shared_keypoints_from_local(orig_record, repl_local_record):
    names = []
    src_pts = []
    dst_pts = []
    for idx, name in enumerate(DOG_KEYPOINT_NAMES):
        if orig_record["confs"][idx] >= KEYPOINT_CONF and repl_local_record["confs"][idx] >= KEYPOINT_CONF:
            names.append(name)
            src_pts.append(repl_local_record["points"][idx])
            dst_pts.append(orig_record["points"][idx])
    if not src_pts:
        return [], np.zeros((0, 2), dtype=np.float32), np.zeros((0, 2), dtype=np.float32)
    return names, np.array(src_pts, dtype=np.float32), np.array(dst_pts, dtype=np.float32)

def transformed_pose_record(pose_record, tform):
    points = np.asarray(tform(pose_record["points"]), dtype=np.float32)
    bbox = pose_record["bbox_xyxy"]
    corners = np.array(
        [
            [bbox[0], bbox[1]],
            [bbox[2], bbox[1]],
            [bbox[2], bbox[3]],
            [bbox[0], bbox[3]],
        ],
        dtype=np.float32,
    )
    warped_corners = np.asarray(tform(corners), dtype=np.float32)
    x1, y1 = warped_corners.min(axis=0)
    x2, y2 = warped_corners.max(axis=0)
    return {
        **pose_record,
        "points": points,
        "bbox_xyxy": [int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))],
    }


def draw_pose_on_image(image, pose_record, color=(80, 220, 120), point_color=None, alpha=0.85, show_bbox=True):
    out = image.copy()
    overlay = out.copy()
    if point_color is None:
        point_color = color
    if pose_record is None:
        return out
    point_map = {
        name: pt
        for name, pt, conf in zip(DOG_KEYPOINT_NAMES, pose_record["points"], pose_record["confs"])
        if conf >= KEYPOINT_CONF
    }
    if show_bbox:
        x1, y1, x2, y2 = pose_record["bbox_xyxy"]
        cv2.rectangle(overlay, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
    for start_name, end_name in DOG_SKELETON_EDGES:
        start = point_map.get(start_name)
        end = point_map.get(end_name)
        if start is None or end is None:
            continue
        cv2.line(
            overlay,
            tuple(int(round(v)) for v in start),
            tuple(int(round(v)) for v in end),
            color,
            2,
            lineType=cv2.LINE_AA,
        )
    for pt in point_map.values():
        center = tuple(int(round(v)) for v in pt)
        cv2.circle(overlay, center, 4, point_color, thickness=-1, lineType=cv2.LINE_AA)
        cv2.circle(overlay, center, 5, (20, 20, 20), thickness=1, lineType=cv2.LINE_AA)
    return ensure_uint8(overlay.astype(np.float32) * alpha + out.astype(np.float32) * (1.0 - alpha))


def draw_pose_comparison(image, pose_items):
    out = image.copy()
    for pose_record, color, point_color, alpha in pose_items:
        out = draw_pose_on_image(out, pose_record, color=color, point_color=point_color, alpha=alpha)
    return out


def rgba_or_rgb_alpha_crop_on_checkerboard(rgb, alpha, mask=None, margin=24):
    if mask is None:
        mask = alpha > 0.01
    bbox = mask_to_bbox(mask)
    if bbox is None:
        return rgba_on_checkerboard(np.dstack([rgb, np.uint8(np.clip(alpha, 0.0, 1.0) * 255)]))
    x1, y1, x2, y2 = bbox
    h, w = alpha.shape[:2]
    x1 = max(0, x1 - margin)
    y1 = max(0, y1 - margin)
    x2 = min(w, x2 + margin)
    y2 = min(h, y2 + margin)
    crop_rgb = rgb[y1:y2, x1:x2]
    crop_alpha = np.uint8(np.clip(alpha[y1:y2, x1:x2], 0.0, 1.0) * 255)
    return rgba_on_checkerboard(np.dstack([crop_rgb, crop_alpha]))



## Batch Pose Compatibility And Warp Processing

For each pair, the notebook now performs the same explainable sequence:

1. load the cutout assets from notebook `01`
2. detect original and replacement dog keypoints
3. decide whether the replacement should be mirrored
4. build explicit keypoint correspondences
5. create a baseline paste and a pose-guided warp
6. export the seed composite and refinement masks for notebook `03`


In [ ]:
print("Pair jobs prepared:", len(PAIR_JOBS))
for job in PAIR_JOBS:
    print(" -", job["pair_id"], job["pair_name"], "->", job["metadata_path"])


## Build Source / Target Correspondences From Shared Keypoints

To keep the warp explainable, the replacement keypoints are explicitly mapped to the original keypoints.

The strategy is:

1. keep only keypoints that are visible in both images
2. solve a similarity transform for a stable whole-body initialization
3. feed the shared keypoints plus the four crop corners into a piecewise-affine warp

If too few keypoints are shared, the notebook falls back to similarity transform only.


In [ ]:
print("Shared-keypoint correspondence is computed inside the batch warp cell below for each pair.")


## Warp And Paste Back: Baseline First, Then Pose-Guided Alignment

The notebook intentionally keeps a very simple baseline for comparison:

- `baseline`: scale and translate only, using box height and bottom alignment
- `pose-guided`: shape-aware warp driven by the detected skeleton

This makes the presentation much clearer because you can show exactly why plain resize-and-paste is not enough, and why pose-guided deformation matters.

Current deformation policy:

- `similarity_only` is the default safe alignment. It uses one global translation, rotation, and scale from shared keypoints.
- `piecewise_affine` is only allowed when there are at least 5 shared keypoints, the similarity fit has `relative_rmse <= 0.12`, and the `00.5` feasibility gate recommends piecewise alignment.
- `micro_piecewise_affine` is used when full piecewise is not allowed but the pair still has at least 4 shared keypoints. It starts from the similarity transform and caps each keypoint's extra local displacement to a small percentage of the original dog bbox diagonal.
- `naive_box_fallback` is used when the similarity fit is too poor, currently `relative_rmse > 0.24`.
- The micro mode has a hard displacement cap. The body and leg default is `max(6 px, 0.035 * original_bbox_diagonal)` per shared keypoint beyond the similarity seed; face/head keypoints use one third of that cap so expression and head geometry are only nudged, not heavily reshaped.

The new diagnostic views below show:

1. original target skeleton vs replacement skeleton after global similarity only
2. original target skeleton vs replacement skeleton after the final selected transform
3. replacement skeleton before and after the final deformation
4. replacement image before and after warping



In [ ]:
def mask_to_bbox(mask):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    return [int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1]


def alpha_blend(background, foreground, alpha):
    alpha = np.clip(alpha[..., None], 0.0, 1.0)
    return ensure_uint8(foreground.astype(np.float32) * alpha + background.astype(np.float32) * (1.0 - alpha))


def read_alpha_float(path):
    return np.asarray(Image.open(path).convert("L"), dtype=np.float32) / 255.0


def apply_soft_alpha_to_rgba(rgba, alpha):
    alpha = np.clip(alpha.astype(np.float32), 0.0, 1.0)
    if alpha.shape != rgba.shape[:2]:
        alpha = cv2.resize(alpha, (rgba.shape[1], rgba.shape[0]), interpolation=cv2.INTER_LINEAR)
    out = rgba.copy()
    out[..., 3] = np.uint8(alpha * 255)
    out[out[..., 3] == 0, :3] = 0
    return out


def estimate_detail_scale_from_warp(source_rgba, placed_mask):
    bbox = mask_to_bbox(placed_mask)
    if bbox is None:
        return 1.0
    x1, y1, x2, y2 = bbox
    warped_w = max(1, x2 - x1)
    warped_h = max(1, y2 - y1)
    src_h, src_w = source_rgba.shape[:2]
    scale_x = src_w / max(1, warped_w)
    scale_y = src_h / max(1, warped_h)
    return float(min(1.0, scale_x, scale_y))


def local_resolution_harmonize(image, focus_mask, detail_scale, margin=72, blur_kernel=31):
    if detail_scale >= 0.98 or focus_mask.sum() < 10:
        return image.copy()
    bbox = mask_to_bbox(focus_mask)
    if bbox is None:
        return image.copy()
    x1, y1, x2, y2 = bbox
    h, w = image.shape[:2]
    x1 = max(0, x1 - margin)
    y1 = max(0, y1 - margin)
    x2 = min(w, x2 + margin)
    y2 = min(h, y2 + margin)
    patch = image[y1:y2, x1:x2].copy()
    local_mask = focus_mask[y1:y2, x1:x2].astype(np.uint8)
    small_w = max(8, int(round(patch.shape[1] * float(detail_scale))))
    small_h = max(8, int(round(patch.shape[0] * float(detail_scale))))
    if small_w >= patch.shape[1] and small_h >= patch.shape[0]:
        return image.copy()
    down = cv2.resize(patch, (small_w, small_h), interpolation=cv2.INTER_AREA)
    up = cv2.resize(down, (patch.shape[1], patch.shape[0]), interpolation=cv2.INTER_LINEAR)
    if blur_kernel % 2 == 0:
        blur_kernel += 1
    soft_mask = cv2.GaussianBlur(local_mask.astype(np.float32), (blur_kernel, blur_kernel), 0)[..., None]
    out_patch = up.astype(np.float32) * soft_mask + patch.astype(np.float32) * (1.0 - soft_mask)
    out = image.copy()
    out[y1:y2, x1:x2] = ensure_uint8(out_patch)
    return out


def expand_hole_mask(mask, expansion_px):
    expansion_px = int(expansion_px)
    if expansion_px <= 0:
        return mask.copy()
    kernel_size = 2 * expansion_px + 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    return cv2.dilate(mask.astype(np.uint8), kernel, iterations=1).astype(bool)


def fast_background_fill(image, hole_mask):
    bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    filled = cv2.inpaint(bgr, hole_mask.astype(np.uint8) * 255, 5, cv2.INPAINT_TELEA)
    return cv2.cvtColor(filled, cv2.COLOR_BGR2RGB)


def build_similarity_seed(src_pts, dst_pts):
    tform = SimilarityTransform()
    if len(src_pts) >= 2 and tform.estimate(src_pts, dst_pts):
        return tform
    raise RuntimeError("Failed to estimate initial similarity transform from shared keypoints.")


def build_rigid_seed_from_size_matched_points(src_pts, dst_pts):
    similarity = build_similarity_seed(src_pts, dst_pts)
    rot_only = SimilarityTransform(scale=1.0, rotation=similarity.rotation, translation=(0.0, 0.0))
    src_center = np.mean(src_pts, axis=0)
    dst_center = np.mean(dst_pts, axis=0)
    translation = dst_center - rot_only(src_center[None, :])[0]
    return SimilarityTransform(scale=1.0, rotation=similarity.rotation, translation=translation)


def build_pose_seed(src_pts, dst_pts, size_prematch_enabled):
    if size_prematch_enabled:
        return build_rigid_seed_from_size_matched_points(src_pts, dst_pts), "rigid_after_size_prematch"
    return build_similarity_seed(src_pts, dst_pts), "similarity_only"


def build_piecewise_or_similarity(src_pts, dst_pts, source_shape, base_seed=None, base_mode="similarity_only"):
    similarity = base_seed if base_seed is not None else build_similarity_seed(src_pts, dst_pts)
    h, w = source_shape[:2]
    src_corners = np.array([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]], dtype=np.float32)
    dst_corners = similarity(src_corners)
    if len(src_pts) >= 4:
        tform = PiecewiseAffineTransform()
        ok = tform.estimate(np.vstack([src_pts, src_corners]), np.vstack([dst_pts, dst_corners]))
        if ok:
            return tform, f"piecewise_affine_from_{base_mode}"
    return similarity, base_mode


class DestinationScaledTransform:
    def __init__(self, base_transform, anchor_xy, scale):
        self.base_transform = base_transform
        self.anchor_xy = np.asarray(anchor_xy, dtype=np.float32)
        self.scale = float(scale)

    def __call__(self, coords):
        pts = np.asarray(self.base_transform(coords), dtype=np.float32)
        return self.anchor_xy + (pts - self.anchor_xy) * self.scale

    def inverse(self, coords):
        coords = np.asarray(coords, dtype=np.float32)
        unscaled = self.anchor_xy + (coords - self.anchor_xy) / max(self.scale, 1e-6)
        return self.base_transform.inverse(unscaled)


def bbox_center_xy(bbox):
    x1, y1, x2, y2 = bbox
    return np.array([(x1 + x2) / 2.0, (y1 + y2) / 2.0], dtype=np.float32)


def apply_destination_scale(tform, target_bbox, scale):
    if abs(float(scale) - 1.0) < 1e-6:
        return tform
    return DestinationScaledTransform(tform, bbox_center_xy(target_bbox), scale)


def bbox_width_height(bbox):
    x1, y1, x2, y2 = bbox
    return max(1.0, float(x2 - x1)), max(1.0, float(y2 - y1))


def bbox_geomean_size(bbox):
    w, h = bbox_width_height(bbox)
    return float(np.sqrt(w * h))


def estimate_replacement_size_prematch_scale(target_bbox, source_bbox):
    target_size = bbox_geomean_size(target_bbox)
    source_size = bbox_geomean_size(source_bbox)
    raw_scale = target_size / max(1.0, source_size)
    clipped_scale = float(np.clip(raw_scale, REPLACEMENT_SIZE_PREMATCH_MIN_SCALE, REPLACEMENT_SIZE_PREMATCH_MAX_SCALE))
    return clipped_scale, raw_scale, target_size, source_size


def resize_rgba_keep_aspect(rgba, scale):
    if abs(float(scale) - 1.0) < 1e-6:
        return rgba.copy()
    h, w = rgba.shape[:2]
    new_w = max(1, int(round(w * float(scale))))
    new_h = max(1, int(round(h * float(scale))))
    rgb = cv2.resize(rgba[..., :3], (new_w, new_h), interpolation=cv2.INTER_CUBIC)
    alpha = cv2.resize(rgba[..., 3], (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    return np.dstack([rgb, alpha.astype(np.uint8)])


def scale_pose_record_local(pose, scale):
    out = dict(pose)
    out["points"] = np.asarray(pose["points"], dtype=np.float32) * float(scale)
    out["bbox_xyxy"] = [int(round(float(v) * float(scale))) for v in pose["bbox_xyxy"]]
    return out


FACE_MICRO_KEYPOINT_NAMES = {
    "nose",
    "chin",
    "left_eye",
    "right_eye",
    "left_ear_base",
    "right_ear_base",
    "left_ear_tip",
    "right_ear_tip",
    "throat",
}


def build_limited_piecewise_transform(
    src_pts,
    dst_pts,
    source_shape,
    similarity,
    max_extra_displacement_px,
    shared_names=None,
    face_cap_scale=1.0 / 3.0,
):
    h, w = source_shape[:2]
    similarity_pts = np.asarray(similarity(src_pts), dtype=np.float32)
    requested_delta = dst_pts.astype(np.float32) - similarity_pts
    delta_norm = np.linalg.norm(requested_delta, axis=1)
    per_keypoint_cap = np.full_like(delta_norm, float(max_extra_displacement_px), dtype=np.float32)

    if shared_names is not None and len(shared_names) == len(per_keypoint_cap):
        face_mask = np.array([name in FACE_MICRO_KEYPOINT_NAMES for name in shared_names], dtype=bool)
        per_keypoint_cap[face_mask] *= float(face_cap_scale)

    scale = np.ones_like(delta_norm, dtype=np.float32)
    large = delta_norm > per_keypoint_cap
    scale[large] = per_keypoint_cap[large] / np.maximum(delta_norm[large], 1e-6)
    limited_dst_pts = similarity_pts + requested_delta * scale[:, None]

    src_corners = np.array([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]], dtype=np.float32)
    dst_corners = similarity(src_corners)
    tform = PiecewiseAffineTransform()
    ok = tform.estimate(np.vstack([src_pts, src_corners]), np.vstack([limited_dst_pts, dst_corners]))
    if not ok:
        return similarity, "similarity_only", limited_dst_pts, float(delta_norm.max()) if len(delta_norm) else 0.0
    return tform, "micro_piecewise_affine", limited_dst_pts, float(delta_norm.max()) if len(delta_norm) else 0.0

def warp_rgba_to_canvas(rgba, tform, output_shape):
    canvas_h, canvas_w = output_shape[:2]
    rgb = rgba[..., :3].astype(np.float32)
    alpha = rgba[..., 3].astype(np.float32) / 255.0
    warped_rgb = warp(
        rgb,
        inverse_map=tform.inverse,
        output_shape=(canvas_h, canvas_w),
        order=1,
        preserve_range=True,
        mode="constant",
        cval=0.0,
    )
    warped_alpha = warp(
        alpha,
        inverse_map=tform.inverse,
        output_shape=(canvas_h, canvas_w),
        order=1,
        preserve_range=True,
        mode="constant",
        cval=0.0,
    )
    warped_mask = warped_alpha > 0.15
    return ensure_uint8(warped_rgb), np.clip(warped_alpha, 0.0, 1.0), warped_mask


def naive_place_rgba(rgba, target_bbox, canvas_shape, destination_scale=1.0):
    x1, y1, x2, y2 = target_bbox
    target_w = max(1, x2 - x1)
    target_h = max(1, y2 - y1)
    src_h, src_w = rgba.shape[:2]
    scale = min(target_w / max(1, src_w), target_h / max(1, src_h)) * float(destination_scale)
    new_w = max(1, int(round(src_w * scale)))
    new_h = max(1, int(round(src_h * scale)))
    resized_rgba = np.dstack(
        [
            cv2.resize(rgba[..., :3], (new_w, new_h), interpolation=cv2.INTER_CUBIC),
            cv2.resize(rgba[..., 3], (new_w, new_h), interpolation=cv2.INTER_LINEAR),
        ]
    )
    left = int(round((x1 + x2) / 2.0 - new_w / 2.0))
    top = int(round(y2 - new_h))
    canvas_h, canvas_w = canvas_shape[:2]
    placed_rgb = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
    placed_alpha = np.zeros((canvas_h, canvas_w), dtype=np.float32)
    x_start = max(0, left)
    y_start = max(0, top)
    x_end = min(canvas_w, left + new_w)
    y_end = min(canvas_h, top + new_h)
    src_x1 = max(0, -left)
    src_y1 = max(0, -top)
    src_x2 = src_x1 + (x_end - x_start)
    src_y2 = src_y1 + (y_end - y_start)
    placed_rgb[y_start:y_end, x_start:x_end] = resized_rgba[src_y1:src_y2, src_x1:src_x2, :3]
    placed_alpha[y_start:y_end, x_start:x_end] = resized_rgba[src_y1:src_y2, src_x1:src_x2, 3].astype(np.float32) / 255.0
    return placed_rgb, placed_alpha, placed_alpha > 0.15



FACE_DRAGGAN_EXCLUDED_BODY_WARP_POINTS = {
    "nose",
    "chin",
    "left_eye",
    "right_eye",
    "left_ear_base",
    "right_ear_base",
    "left_ear_tip",
    "right_ear_tip",
    "throat",
}


def load_true_draggan_face_package(pair_name):
    edit_meta_path = FACE_EDIT_ROOT / pair_name / "metadata.json"
    if not edit_meta_path.exists():
        raise FileNotFoundError(f"True DragGAN face edit metadata not found at {edit_meta_path}. Run notebooks 02 and 03 first.")
    with open(edit_meta_path, "r", encoding="utf-8") as f:
        edit_meta = json.load(f)
    inversion_meta_path = PROJECT_ROOT / edit_meta["inversion_metadata_path"]
    with open(inversion_meta_path, "r", encoding="utf-8") as f:
        inversion_meta = json.load(f)
    edited_face = read_rgb(PROJECT_ROOT / edit_meta["outputs"]["draggan_edited_face_512"])
    edit_meta["selected_face_source"] = "03_true_draggan_face_edit"
    edit_meta["selected_face_metadata_path"] = str(edit_meta_path)
    return edit_meta, inversion_meta, edited_face


def paste_true_draggan_face_into_rgba(rep_rgba, edited_face_512, face_crop_raw_box):
    x1, y1, x2, y2 = [int(v) for v in face_crop_raw_box]
    h, w = rep_rgba.shape[:2]
    side = max(1, x2 - x1, y2 - y1)
    resized_face = cv2.resize(edited_face_512, (side, side), interpolation=cv2.INTER_CUBIC)
    output = rep_rgba.copy()

    crop_x1 = max(0, x1)
    crop_y1 = max(0, y1)
    crop_x2 = min(w, x2)
    crop_y2 = min(h, y2)
    if crop_x2 <= crop_x1 or crop_y2 <= crop_y1:
        raise RuntimeError("Edited face crop does not overlap replacement cutout.")

    src_x1 = crop_x1 - x1
    src_y1 = crop_y1 - y1
    src_x2 = src_x1 + (crop_x2 - crop_x1)
    src_y2 = src_y1 + (crop_y2 - crop_y1)
    face_patch = resized_face[src_y1:src_y2, src_x1:src_x2]

    local_h, local_w = face_patch.shape[:2]
    ellipse = np.zeros((local_h, local_w), dtype=np.float32)
    cv2.ellipse(
        ellipse,
        (local_w // 2, local_h // 2),
        (max(1, int(local_w * 0.46)), max(1, int(local_h * 0.48))),
        0,
        0,
        360,
        1.0,
        -1,
    )
    ellipse = cv2.GaussianBlur(ellipse, (31, 31), 0)
    alpha = np.clip(ellipse[..., None], 0.0, 1.0)
    original_rgb = output[crop_y1:crop_y2, crop_x1:crop_x2, :3].astype(np.float32)
    blended = face_patch.astype(np.float32) * alpha + original_rgb * (1.0 - alpha)
    output[crop_y1:crop_y2, crop_x1:crop_x2, :3] = np.clip(blended, 0, 255).astype(np.uint8)
    return output


def keep_body_only_shared_keypoints(shared_names, src_pts, dst_pts):
    keep = [i for i, name in enumerate(shared_names) if name not in FACE_DRAGGAN_EXCLUDED_BODY_WARP_POINTS]
    if len(keep) < 3:
        raise RuntimeError("Too few body/leg keypoints after excluding face/head keypoints. This pair should not use the true-DragGAN face branch.")
    return [shared_names[i] for i in keep], src_pts[keep], dst_pts[keep]


def build_refinement_masks(hole_mask, placed_mask):
    residual_hole_mask = np.logical_and(hole_mask, ~placed_mask)
    seam_outer = cv2.dilate(placed_mask.astype(np.uint8), np.ones((31, 31), np.uint8), iterations=1).astype(bool)
    seam_inner = cv2.dilate(placed_mask.astype(np.uint8), np.ones((9, 9), np.uint8), iterations=1).astype(bool)
    seam_ring = np.logical_and(seam_outer, ~seam_inner)
    refine = np.logical_or(residual_hole_mask, seam_ring)
    refine = np.logical_and(refine, ~placed_mask)
    refine = cv2.morphologyEx(refine.astype(np.uint8), cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8)).astype(bool)
    residual_hole_mask = cv2.morphologyEx(residual_hole_mask.astype(np.uint8), cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8)).astype(bool)
    return refine, residual_hole_mask, seam_ring


WARP_RESULTS = []

for pair_index, job in enumerate(PAIR_JOBS, start=1):
    with open(job["metadata_path"], "r", encoding="utf-8") as f:
        cutout_meta = json.load(f)

    original_img = read_rgb(Path(cutout_meta["inputs"]["original_path"]))
    replacement_img = read_rgb(Path(cutout_meta["inputs"]["replacement_path"]))
    org_hole_mask_raw = read_mask(PROJECT_ROOT / cutout_meta["original"]["hole_mask_path"])
    org_hole_mask = expand_hole_mask(org_hole_mask_raw, BACKGROUND_HOLE_EXPANSION_PX)
    org_mask = read_mask(PROJECT_ROOT / cutout_meta["original"]["mask_path"])
    org_background_with_hole = read_rgb(PROJECT_ROOT / cutout_meta["original"]["background_with_hole_path"])
    rep_rgba = read_rgba(PROJECT_ROOT / cutout_meta["replacement"]["rgba_path"])
    replacement_alpha_source = "rgba_embedded_alpha"
    soft_alpha_rel = cutout_meta["replacement"].get("soft_alpha_path")
    if soft_alpha_rel:
        soft_alpha_path = PROJECT_ROOT / soft_alpha_rel
        if soft_alpha_path.exists():
            rep_rgba = apply_soft_alpha_to_rgba(rep_rgba, read_alpha_float(soft_alpha_path))
            replacement_alpha_source = "soft_alpha_path"
    true_draggan_edit_meta, true_draggan_inversion_meta, true_draggan_face_512 = load_true_draggan_face_package(job["pair_name"])
    rep_rgba = paste_true_draggan_face_into_rgba(rep_rgba, true_draggan_face_512, true_draggan_inversion_meta["face_crop_raw_box_xyxy"])
    rep_crop_bbox = cutout_meta["replacement"]["crop_bbox_xyxy"]
    approval = cutout_meta.get("approval") or {}

    print_pair_header(f"[02] Pair {pair_index}/{len(PAIR_JOBS)} :: {job['pair_name']}")
    print("Cutout metadata:", job["metadata_path"])
    print("Original image shape:", original_img.shape)
    print("Replacement image shape:", replacement_img.shape)
    print("Replacement crop shape:", rep_rgba.shape)
    print("Replacement alpha source:", replacement_alpha_source)
    print("Selected face source:", true_draggan_edit_meta.get("selected_face_source"))

    plot_images(
        [
            ("Original image", original_img),
            ("Original hole mask from 01", org_hole_mask_raw),
            ("Expanded hole mask for 02 background fill", org_hole_mask),
            ("Original background with hole", org_background_with_hole),
            ("Replacement image", replacement_img),
            ("Replacement RGBA on checkerboard", rgba_on_checkerboard(rep_rgba)),
            ("Replacement soft alpha used by body warp", rep_rgba[..., 3]),
        ],
        cols=2,
        figsize=(16, 12),
    )

    org_pose_local = deserialize_pose_record(cutout_meta["original"].get("cutout_pose"))
    repl_pose_local = deserialize_pose_record(cutout_meta["replacement"].get("cutout_pose"))
    if org_pose_local is None or repl_pose_local is None:
        raise RuntimeError(f"Cutout pose metadata missing for {job['pair_name']}. Re-run notebook 01.")

    org_crop_bbox = cutout_meta["original"]["crop_bbox_xyxy"]
    orig_pose = translate_pose_record(org_pose_local, dx=org_crop_bbox[0], dy=org_crop_bbox[1])
    repl_pose_canonical = repl_pose_local

    mirror_replacement = (
        orig_pose["view_class"] == "side_profile"
        and repl_pose_local["view_class"] == "side_profile"
        and orig_pose["orientation"] in {"left", "right"}
        and repl_pose_local["orientation"] in {"left", "right"}
        and orig_pose["orientation"] != repl_pose_local["orientation"]
    )

    replacement_img_canonical = replacement_img[:, ::-1].copy() if mirror_replacement else replacement_img.copy()
    rep_rgba_canonical = rep_rgba[:, ::-1].copy() if mirror_replacement else rep_rgba.copy()
    if mirror_replacement:
        repl_points = repl_pose_local["points"].copy()
        repl_points[:, 0] = (rep_rgba.shape[1] - 1) - repl_points[:, 0]
        repl_bbox = repl_pose_local["bbox_xyxy"]
        repl_pose_canonical = {
            **repl_pose_local,
            "points": repl_points,
            "bbox_xyxy": [
                int(round((rep_rgba.shape[1] - 1) - repl_bbox[2])),
                repl_bbox[1],
                int(round((rep_rgba.shape[1] - 1) - repl_bbox[0])),
                repl_bbox[3],
            ],
            "orientation": orig_pose["orientation"],
        }

    org_size_bbox = mask_to_bbox(org_mask)
    if org_size_bbox is None:
        org_size_bbox = cutout_meta["original"]["bbox_xyxy"]
    rep_size_bbox = mask_to_bbox(rep_rgba_canonical[..., 3] > 15)
    if rep_size_bbox is None:
        rep_size_bbox = repl_pose_canonical["bbox_xyxy"]

    size_prematch_scale = 1.0
    size_prematch_raw_scale = 1.0
    org_size_measure = bbox_geomean_size(org_size_bbox)
    rep_size_measure_before = bbox_geomean_size(rep_size_bbox)
    rep_rgba_before_size_match = rep_rgba_canonical.copy()
    repl_pose_before_size_match = repl_pose_canonical
    if ENABLE_REPLACEMENT_SIZE_PREMATCH:
        size_prematch_scale, size_prematch_raw_scale, org_size_measure, rep_size_measure_before = estimate_replacement_size_prematch_scale(
            org_size_bbox,
            rep_size_bbox,
        )
        rep_rgba_canonical = resize_rgba_keep_aspect(rep_rgba_canonical, size_prematch_scale)
        repl_pose_canonical = scale_pose_record_local(repl_pose_canonical, size_prematch_scale)

    rep_size_bbox_after = mask_to_bbox(rep_rgba_canonical[..., 3] > 15) or repl_pose_canonical["bbox_xyxy"]
    rep_size_measure_after = bbox_geomean_size(rep_size_bbox_after)

    print("Original cutout pose:", {k: orig_pose[k] for k in ['view_class', 'orientation', 'visible_keypoints', 'bbox_xyxy']})
    print("Replacement cutout pose before size pre-match:", {k: repl_pose_before_size_match[k] for k in ['view_class', 'orientation', 'visible_keypoints', 'bbox_xyxy']})
    print("Replacement cutout pose after size pre-match:", {k: repl_pose_canonical[k] for k in ['view_class', 'orientation', 'visible_keypoints', 'bbox_xyxy']})
    print("Mirror replacement before warp:", mirror_replacement)
    print("Replacement size pre-match scale:", round(float(size_prematch_scale), 4), "| raw:", round(float(size_prematch_raw_scale), 4))
    print("Size measure org / rep before / rep after:", round(float(org_size_measure), 2), round(float(rep_size_measure_before), 2), round(float(rep_size_measure_after), 2))

    draw_pose_overlay(original_img, orig_pose, title=f"Original cutout pose mapped to full image :: {job['pair_name']}")
    draw_pose_overlay(rgba_on_checkerboard(rep_rgba_before_size_match), repl_pose_before_size_match, title=f"Replacement pose before size pre-match :: {job['pair_name']}")
    draw_pose_overlay(rgba_on_checkerboard(rep_rgba_canonical), repl_pose_canonical, title=f"Replacement pose after size pre-match :: {job['pair_name']}")

    shared_names_all, src_pts_all, dst_pts_all = shared_keypoints_from_local(orig_pose, repl_pose_canonical)
    shared_names, src_pts, dst_pts = keep_body_only_shared_keypoints(shared_names_all, src_pts_all, dst_pts_all)
    print("Body/leg keypoint count after excluding face:", len(shared_names))
    print("Shared keypoint names:", shared_names)
    if len(shared_names) < 3:
        raise RuntimeError(f"Too few shared keypoints for {job['pair_name']}.")

    similarity_seed, base_transform_mode = build_pose_seed(src_pts, dst_pts, ENABLE_REPLACEMENT_SIZE_PREMATCH)
    similarity_rmse = pose_rmse(src_pts, dst_pts, similarity_seed)
    orig_bbox = cutout_meta["original"]["bbox_xyxy"]
    orig_diag = float(np.linalg.norm([orig_bbox[2] - orig_bbox[0], orig_bbox[3] - orig_bbox[1]]))
    relative_rmse = similarity_rmse / max(1.0, orig_diag)

    recommended_mode = approval.get("recommended_transform_mode", "similarity_only")
    allow_piecewise = bool(ENABLE_FULL_PIECEWISE_WARP) and len(shared_names) >= 5 and relative_rmse <= 0.12 and recommended_mode == "piecewise_affine"
    allow_micro_piecewise = bool(ENABLE_MICRO_PIECEWISE_WARP) and len(shared_names) >= 4 and relative_rmse <= 0.24
    micro_max_extra_displacement_px = max(float(MICRO_PIECEWISE_MIN_EXTRA_PX), float(MICRO_PIECEWISE_MAX_EXTRA_REL) * orig_diag)
    micro_face_extra_displacement_px = micro_max_extra_displacement_px / 3.0
    force_naive = relative_rmse > 0.24

    similarity_pose = transformed_pose_record(repl_pose_canonical, similarity_seed)

    if BODY_ALIGNMENT_MODE == "face_only_naive_box":
        transform_mode = "face_draggan_only_naive_box"
        tform = None
        final_pose = None
        allow_piecewise = False
        allow_micro_piecewise = False
        force_naive = True
        warped_rgb, warped_alpha, warped_mask = naive_place_rgba(
            rep_rgba_canonical,
            orig_bbox,
            original_img.shape,
            destination_scale=REPLACEMENT_DESTINATION_SCALE,
        )
        extra_warp_mean_px = 0.0
        extra_warp_max_px = 0.0
        extra_warp_mean_rel = 0.0
        extra_warp_max_rel = 0.0
    elif force_naive:
        transform_mode = "naive_box_fallback"
        tform = None
        final_pose = None
        warped_rgb, warped_alpha, warped_mask = naive_place_rgba(
            rep_rgba_canonical,
            orig_bbox,
            original_img.shape,
            destination_scale=REPLACEMENT_DESTINATION_SCALE,
        )
        extra_warp_mean_px = 0.0
        extra_warp_max_px = 0.0
        extra_warp_mean_rel = 0.0
        extra_warp_max_rel = 0.0
    else:
        if allow_piecewise:
            tform, transform_mode = build_piecewise_or_similarity(
                src_pts,
                dst_pts,
                rep_rgba_canonical.shape,
                base_seed=similarity_seed,
                base_mode=base_transform_mode,
            )
            final_pts_for_metrics = np.asarray(tform(src_pts), dtype=np.float32)
        elif allow_micro_piecewise:
            tform, transform_mode, final_pts_for_metrics, requested_extra_max_px = build_limited_piecewise_transform(
                src_pts,
                dst_pts,
                rep_rgba_canonical.shape,
                similarity_seed,
                micro_max_extra_displacement_px,
                shared_names=shared_names,
            )
        else:
            tform, transform_mode = similarity_seed, base_transform_mode
            final_pts_for_metrics = np.asarray(tform(src_pts), dtype=np.float32)
        if abs(REPLACEMENT_DESTINATION_SCALE - 1.0) > 1e-6:
            tform = apply_destination_scale(tform, orig_bbox, REPLACEMENT_DESTINATION_SCALE)
            transform_mode = f"{transform_mode}+dest_scale_{REPLACEMENT_DESTINATION_SCALE:.3f}"
            final_pts_for_metrics = np.asarray(tform(src_pts), dtype=np.float32)
        final_pose = transformed_pose_record(repl_pose_canonical, tform)
        warped_rgb, warped_alpha, warped_mask = warp_rgba_to_canvas(rep_rgba_canonical, tform, original_img.shape)
        similarity_pts = np.asarray(similarity_seed(src_pts), dtype=np.float32)
        final_pts = np.asarray(final_pts_for_metrics, dtype=np.float32)
        extra_displacement = np.linalg.norm(final_pts - similarity_pts, axis=1)
        extra_warp_mean_px = float(extra_displacement.mean()) if len(extra_displacement) else 0.0
        extra_warp_max_px = float(extra_displacement.max()) if len(extra_displacement) else 0.0
        extra_warp_mean_rel = extra_warp_mean_px / max(1.0, orig_diag)
        extra_warp_max_rel = extra_warp_max_px / max(1.0, orig_diag)

    seed_background = fast_background_fill(original_img, org_hole_mask)
    seed_composite = alpha_blend(seed_background, warped_rgb, warped_alpha)
    rep_detail_scale = estimate_detail_scale_from_warp(rep_rgba_canonical, warped_mask)
    resolution_focus_mask = cv2.dilate(
        np.logical_or(warped_mask, org_hole_mask).astype(np.uint8),
        np.ones((31, 31), np.uint8),
        iterations=1,
    ).astype(bool)
    resolution_harmonized_seed = local_resolution_harmonize(
        seed_composite,
        resolution_focus_mask,
        rep_detail_scale,
        margin=72,
        blur_kernel=31,
    )

    baseline_rgb, baseline_alpha, baseline_mask = naive_place_rgba(
        rep_rgba_canonical,
        orig_bbox,
        original_img.shape,
        destination_scale=REPLACEMENT_DESTINATION_SCALE,
    )
    baseline_composite = alpha_blend(seed_background, baseline_rgb, baseline_alpha)

    refinement_mask, residual_hole_mask, seam_ring_mask = build_refinement_masks(org_hole_mask, warped_mask)

    print("Recommended transform mode from 00.5:", recommended_mode)
    print("Similarity RMSE:", round(similarity_rmse, 4))
    print("Relative RMSE:", round(relative_rmse, 4))
    print("Transform mode used:", transform_mode)
    print("Replacement size pre-match enabled:", bool(ENABLE_REPLACEMENT_SIZE_PREMATCH))
    print("Replacement size pre-match scale:", round(float(size_prematch_scale), 4))
    print("Replacement destination scale:", round(float(REPLACEMENT_DESTINATION_SCALE), 4))
    print("Background hole expansion px:", int(BACKGROUND_HOLE_EXPANSION_PX))
    print("Original hole mask pixels:", int(org_hole_mask_raw.sum()))
    print("Expanded hole mask pixels:", int(org_hole_mask.sum()))
    print("Body alignment mode:", BODY_ALIGNMENT_MODE)
    print("Enable full piecewise warp:", bool(ENABLE_FULL_PIECEWISE_WARP))
    print("Enable micro-piecewise warp:", bool(ENABLE_MICRO_PIECEWISE_WARP))
    print("Piecewise allowed:", bool(allow_piecewise))
    print("Micro piecewise allowed:", bool(allow_micro_piecewise))
    print("Micro body/leg max extra displacement px:", round(float(micro_max_extra_displacement_px), 4))
    print("Micro face/head max extra displacement px:", round(float(micro_face_extra_displacement_px), 4))
    print("Force naive fallback:", bool(force_naive))
    print("Extra warp mean displacement px:", round(extra_warp_mean_px, 4))
    print("Extra warp max displacement px:", round(extra_warp_max_px, 4))
    print("Extra warp mean displacement / original bbox diagonal:", round(extra_warp_mean_rel, 4))
    print("Extra warp max displacement / original bbox diagonal:", round(extra_warp_max_rel, 4))
    print("Warped mask pixels:", int(warped_mask.sum()))
    print("Residual hole mask pixels:", int(residual_hole_mask.sum()))
    print("Refinement mask pixels:", int(refinement_mask.sum()))
    print("Replacement effective detail scale:", round(rep_detail_scale, 4))

    if final_pose is None:
        final_pose_for_display = similarity_pose
    else:
        final_pose_for_display = final_pose

    pose_before_fit = draw_pose_comparison(
        original_img,
        [
            (orig_pose, (80, 230, 120), (230, 255, 120), 0.95),
            (similarity_pose, (255, 165, 60), (255, 210, 80), 0.70),
        ],
    )
    pose_after_fit = draw_pose_comparison(
        original_img,
        [
            (orig_pose, (80, 230, 120), (230, 255, 120), 0.95),
            (final_pose_for_display, (190, 95, 255), (235, 190, 255), 0.78),
        ],
    )
    rep_pose_before_after = draw_pose_comparison(
        original_img,
        [
            (similarity_pose, (255, 165, 60), (255, 210, 80), 0.70),
            (final_pose_for_display, (190, 95, 255), (235, 190, 255), 0.78),
        ],
    )
    warped_replacement_checker = rgba_or_rgb_alpha_crop_on_checkerboard(warped_rgb, warped_alpha, warped_mask, margin=24)

    plot_images(
        [
            ("1. Skeletons before final deformation: org target + rep similarity seed", pose_before_fit),
            ("2. Skeletons after final deformation: org target + rep fitted pose", pose_after_fit),
            ("3. Replacement skeleton movement: similarity seed + final fitted pose", rep_pose_before_after),
            ("4a. Replacement image before size pre-match", rgba_on_checkerboard(rep_rgba_before_size_match)),
            ("4b. Replacement image after size pre-match", rgba_on_checkerboard(rep_rgba_canonical)),
            ("4c. Replacement image after warp", warped_replacement_checker),
        ],
        cols=2,
        figsize=(18, 16),
    )

    plot_images(
        [
            ("Fast background seed", seed_background),
            ("Baseline composite (bbox only)", baseline_composite),
            ("Pose-guided warped mask", warped_mask),
            ("Residual uncovered hole mask", residual_hole_mask),
            ("Seam ring mask", seam_ring_mask),
            ("Pose-guided seed composite", seed_composite),
            ("Resolution-harmonized seed", resolution_harmonized_seed),
            ("Refinement mask for notebook 03", refinement_mask),
        ],
        cols=3,
        figsize=(18, 14),
    )

    output_dir = job["output_dir"]
    output_dir.mkdir(parents=True, exist_ok=True)
    save_rgb(output_dir / "seed_background.png", seed_background)
    save_rgb(output_dir / "baseline_composite.png", baseline_composite)
    save_rgb(output_dir / "seed_composite.png", seed_composite)
    save_rgb(output_dir / "resolution_harmonized_seed.png", resolution_harmonized_seed)
    save_mask(output_dir / "original_hole_mask_raw.png", org_hole_mask_raw)
    save_mask(output_dir / "original_hole_mask_expanded.png", org_hole_mask)
    save_mask(output_dir / "warped_mask.png", warped_mask)
    save_mask(output_dir / "refinement_mask.png", refinement_mask)
    save_mask(output_dir / "residual_hole_mask.png", residual_hole_mask)
    save_mask(output_dir / "seam_ring_mask.png", seam_ring_mask)
    save_rgb(output_dir / "warped_rgb.png", warped_rgb)
    save_rgb(output_dir / "pose_before_final_deformation.png", pose_before_fit)
    save_rgb(output_dir / "pose_after_final_deformation.png", pose_after_fit)
    save_rgb(output_dir / "rep_pose_before_after_deformation.png", rep_pose_before_after)
    save_rgb(output_dir / "warped_replacement_checker.png", warped_replacement_checker)
    Image.fromarray((np.clip(warped_alpha, 0.0, 1.0) * 255).astype(np.uint8)).save(output_dir / "warped_alpha.png")
    Image.fromarray((np.clip(warped_alpha, 0.0, 1.0) * 255).astype(np.uint8)).save(output_dir / "warped_soft_alpha.png")

    warp_meta = {
        "pair_id": job["pair_id"],
        "pair_name": job["pair_name"],
        "pair_index": pair_index,
        "cutout_metadata_path": str(job["metadata_path"]),
        "pose_model_path": str(POSE_MODEL_PATH),
        "approval": approval,
        "mirror_replacement": bool(mirror_replacement),
        "transform_mode": transform_mode,
        "recommended_transform_mode": recommended_mode,
        "similarity_rmse": round(float(similarity_rmse), 6),
        "relative_rmse": round(float(relative_rmse), 6),
        "replacement_effective_detail_scale": round(float(rep_detail_scale), 6),
        "replacement_size_prematch_enabled": bool(ENABLE_REPLACEMENT_SIZE_PREMATCH),
        "replacement_size_prematch_scale": round(float(size_prematch_scale), 6),
        "replacement_size_prematch_raw_scale": round(float(size_prematch_raw_scale), 6),
        "original_size_match_bbox": [int(v) for v in org_size_bbox],
        "replacement_size_match_bbox_before": [int(v) for v in rep_size_bbox],
        "replacement_size_match_bbox_after": [int(v) for v in rep_size_bbox_after],
        "original_size_measure": round(float(org_size_measure), 6),
        "replacement_size_measure_before": round(float(rep_size_measure_before), 6),
        "replacement_size_measure_after": round(float(rep_size_measure_after), 6),
        "replacement_destination_scale": round(float(REPLACEMENT_DESTINATION_SCALE), 6),
        "replacement_alpha_source": replacement_alpha_source,
        "background_hole_expansion_px": int(BACKGROUND_HOLE_EXPANSION_PX),
        "true_draggan_face_source": "03_true_draggan_face_edit",
        "true_draggan_face_metadata_path": str(FACE_EDIT_ROOT / job["pair_name"] / "metadata.json"),
        "true_draggan_face_edit_metadata_path": str(FACE_EDIT_ROOT / job["pair_name"] / "metadata.json"),
        "body_alignment_mode": BODY_ALIGNMENT_MODE,
        "body_warp_excludes_face_keypoints": bool(BODY_ALIGNMENT_MODE == "body_warp"),
        "original_hole_mask_pixels": int(org_hole_mask_raw.sum()),
        "expanded_hole_mask_pixels": int(org_hole_mask.sum()),
        "enable_full_piecewise_warp": bool(ENABLE_FULL_PIECEWISE_WARP),
        "enable_micro_piecewise_warp": bool(ENABLE_MICRO_PIECEWISE_WARP),
        "allow_piecewise": bool(allow_piecewise),
        "allow_micro_piecewise": bool(allow_micro_piecewise),
        "micro_piecewise_max_extra_rel": round(float(MICRO_PIECEWISE_MAX_EXTRA_REL), 6),
        "micro_max_extra_displacement_px": round(float(micro_max_extra_displacement_px), 6),
        "micro_face_extra_displacement_px": round(float(micro_face_extra_displacement_px), 6),
        "force_naive_fallback": bool(force_naive),
        "extra_warp_mean_displacement_px": round(float(extra_warp_mean_px), 6),
        "extra_warp_max_displacement_px": round(float(extra_warp_max_px), 6),
        "extra_warp_mean_displacement_rel": round(float(extra_warp_mean_rel), 6),
        "extra_warp_max_displacement_rel": round(float(extra_warp_max_rel), 6),
        "shared_keypoint_names": shared_names,
        "original_pose": {
            "view_class": orig_pose["view_class"],
            "orientation": orig_pose["orientation"],
            "visible_keypoints": int(orig_pose["visible_keypoints"]),
            "bbox_xyxy": [int(v) for v in orig_pose["bbox_xyxy"]],
        },
        "replacement_pose": {
            "view_class": repl_pose_canonical["view_class"],
            "orientation": repl_pose_canonical["orientation"],
            "visible_keypoints": int(repl_pose_canonical["visible_keypoints"]),
            "bbox_xyxy": [int(v) for v in repl_pose_canonical["bbox_xyxy"]],
            "size_prematched": bool(ENABLE_REPLACEMENT_SIZE_PREMATCH),
            "size_prematch_scale": round(float(size_prematch_scale), 6),
        },
        "outputs": {
            "seed_background": str(output_dir / "seed_background.png"),
            "baseline_composite": str(output_dir / "baseline_composite.png"),
            "seed_composite": str(output_dir / "seed_composite.png"),
            "resolution_harmonized_seed": str(output_dir / "resolution_harmonized_seed.png"),
            "original_hole_mask_raw": str(output_dir / "original_hole_mask_raw.png"),
            "original_hole_mask_expanded": str(output_dir / "original_hole_mask_expanded.png"),
            "warped_mask": str(output_dir / "warped_mask.png"),
            "refinement_mask": str(output_dir / "refinement_mask.png"),
            "residual_hole_mask": str(output_dir / "residual_hole_mask.png"),
            "seam_ring_mask": str(output_dir / "seam_ring_mask.png"),
            "warped_rgb": str(output_dir / "warped_rgb.png"),
            "pose_before_final_deformation": str(output_dir / "pose_before_final_deformation.png"),
            "pose_after_final_deformation": str(output_dir / "pose_after_final_deformation.png"),
            "rep_pose_before_after_deformation": str(output_dir / "rep_pose_before_after_deformation.png"),
            "warped_replacement_checker": str(output_dir / "warped_replacement_checker.png"),
            "warped_alpha": str(output_dir / "warped_alpha.png"),
            "warped_soft_alpha": str(output_dir / "warped_soft_alpha.png"),
        },
    }
    with open(output_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(warp_meta, f, ensure_ascii=False, indent=2)

    print("Saved warp package to:", output_dir)
    print("Metadata file:", output_dir / "metadata.json")

    plot_images(
        [
            ("Original hole background from 01", org_background_with_hole),
            ("Expanded hole mask used by 02", org_hole_mask),
            ("Fast background seed from expanded hole", seed_background),
            ("Pose-guided seed composite", seed_composite),
            ("Residual hole mask", residual_hole_mask),
            ("Refinement mask", refinement_mask),
        ],
        cols=3,
        figsize=(18, 10),
    )

    WARP_RESULTS.append(
        {
            "pair_name": job["pair_name"],
            "pair_index": pair_index,
            "output_dir": str(output_dir),
            "metadata_path": str(output_dir / "metadata.json"),
            "transform_mode": transform_mode,
        }
    )

with open(OUTPUT_ROOT / "batch_manifest.json", "w", encoding="utf-8") as f:
    json.dump(WARP_RESULTS, f, ensure_ascii=False, indent=2)


## Export The Assets Needed By 03

This notebook now exports two related repair masks:

- `refinement_mask`: the broad local-refinement mask used by notebook `03`
- `residual_hole_mask`: the specific region where the pasted replacement dog still fails to cover the original removed-dog hole

That second mask is useful because it targets the exact failure mode you called out: if the replacement silhouette is slightly too small or the cutout edge is a little inaccurate, notebook `03` can widen that region a little and repair it more aggressively without over-expanding the whole refinement area.


In [ ]:
with open(OUTPUT_ROOT / "batch_manifest.json", "w", encoding="utf-8") as f:
    json.dump(WARP_RESULTS, f, ensure_ascii=False, indent=2)

print("Processed warp pairs:")
for item in WARP_RESULTS:
    print(" -", item["pair_index"], item["pair_name"], "|", item["transform_mode"], "->", item["output_dir"])
print("Saved batch manifest:", OUTPUT_ROOT / "batch_manifest.json")


In [ ]:
pass
